<a href="https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — The Age-Freshness Matrix

Where does the label come from: Health score per age×freshness cell, computed from the active-content sample (61.8K rows), not the full 341K portfolio.

Methodology question: The paper flags survivor bias in the 365+ × 361+ cell (only 1 declining page). If that cell has this problem, do the other sparse cells in the matrix (e.g. 181-365 × 31-90, which also looks like a small sample) carry the same issue without being flagged? How many rows actually sit in each cell?

Finding 2 — AI Model Performance (OpenAI vs Gemini)

Where does the label come from: Health score compared across provider cohorts (OpenAI 145.5K vs Gemini 91.4K rows), split by age tier.

Methodology question: The methodology section reports no p-values or confidence intervals anywhere in the paper. Without significance testing, how do we know the OpenAI-vs-Gemini health gap (18.92 vs 27.13) isn't just noise from unequal sample sizes or unmeasured confounders, rather than a real provider difference?

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

To show the value of the grouped split, I compare it against a random row split on the same clean features (no leaky columns). If a client's pages leak between train and test, the random split should look artificially better.

Result: random split scores 0.86 precision@50 vs 0.74 for the grouped split
— a 12-point gap. This confirms the random split was overstating model
performance: it let the model partly recognize clients it had already
seen in training, instead of testing on genuinely unseen clients. The
grouped-by-client_id number (0.74) is the honest one to report.

In [8]:
%cd /content
!rm -rf flyrank-ml
!git clone https://github.com/uomna/flyrank-ml.git
%cd flyrank-ml
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

df_valid = df[df["avg_position"] > 0].copy()

# honest split: grouped by client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_valid, groups=df_valid["client_id"]))
train_df = df_valid.iloc[train_idx]
test_df = df_valid.iloc[test_idx].copy()

feature_cols_clean = [
    "search_volume", "competition", "cpc", "word_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

X_train_c = train_df[feature_cols_clean]
X_test_c = test_df[feature_cols_clean]
y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

imputer_c = SimpleImputer(strategy="median")
X_train_c_imp = imputer_c.fit_transform(X_train_c)
X_test_c_imp = imputer_c.transform(X_test_c)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c_imp)
X_test_c_scaled = scaler_c.transform(X_test_c_imp)

log_reg_clean = LogisticRegression(max_iter=1000, random_state=42)
log_reg_clean.fit(X_train_c_scaled, y_train)

test_df["model_score_clean"] = log_reg_clean.predict_proba(X_test_c_scaled)[:, 1]

def precision_at_k(frame, score_col, k=50):
    top_k = frame.sort_values(score_col, ascending=False).head(k)
    return top_k["is_declining_label"].mean()

model_clean_p50 = precision_at_k(test_df, "model_score_clean", 50)
print("Grouped split precision@50:", model_clean_p50)

/content
Cloning into 'flyrank-ml'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 162 (delta 65), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 1.88 MiB | 6.73 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/content/flyrank-ml
Grouped split precision@50: 0.74


In [9]:
from sklearn.model_selection import train_test_split

# random split — same rows (df_valid), NOT grouped by client
train_r, test_r = train_test_split(
    df_valid, test_size=0.2, random_state=42, stratify=df_valid["is_declining_label"]
)

X_train_r = train_r[feature_cols_clean]
X_test_r = test_r[feature_cols_clean]
y_train_r = train_r["is_declining_label"]
y_test_r = test_r["is_declining_label"]

imputer_r = SimpleImputer(strategy="median")
X_train_r_imp = imputer_r.fit_transform(X_train_r)
X_test_r_imp = imputer_r.transform(X_test_r)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r_imp)
X_test_r_scaled = scaler_r.transform(X_test_r_imp)

log_reg_random = LogisticRegression(max_iter=1000, random_state=42)
log_reg_random.fit(X_train_r_scaled, y_train_r)

test_r = test_r.copy()
test_r["model_score_random"] = log_reg_random.predict_proba(X_test_r_scaled)[:, 1]

random_p50 = precision_at_k(test_r, "model_score_random", 50)

split_comparison = pd.DataFrame({
    "split_type": ["random split (dishonest)", "grouped by client_id (honest)"],
    "precision_at_50": [random_p50, model_clean_p50]
})
print(split_comparison)

                      split_type  precision_at_50
0       random split (dishonest)             0.86
1  grouped by client_id (honest)             0.74


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same hunt as Week 3/5: check whether any remaining feature is suspiciously
dominant, which would suggest it's a sibling of the label rather than a
genuine signal.
No feature dominates the way impressions_last_30d/prev_30d did before
removal (coefficient 35.5). The top feature now is users_90d at 1.17 —
roughly 30x smaller. This spread across 10 features, none of them a
sibling of trend_direction/trend_pct, suggests the model is combining
several weak-to-moderate signals rather than reading the label off one
column.

Checklist:
- [x] No trend_direction / trend_pct in features
- [x] No id columns (content_id, client_id) used as features
- [x] No single feature >10x the next largest coefficient
- [x] Train-without test already run in Week 5 (leaky removal dropped
      precision@50 from 1.00 to 0.74 — the confession)

In [10]:
coef_table_final = pd.DataFrame({
    "feature": feature_cols_clean,
    "coefficient": log_reg_clean.coef_[0]
})
coef_table_final["abs_coefficient"] = coef_table_final["coefficient"].abs()
coef_table_final = coef_table_final.sort_values("abs_coefficient", ascending=False)
print(coef_table_final.head(10))

                  feature  coefficient  abs_coefficient
8               users_90d    -1.165402         1.165402
7            sessions_90d     0.958395         0.958395
14       content_age_days    -0.387111         0.387111
12  days_with_impressions     0.379026         0.379026
13     days_with_sessions    -0.365935         0.365935
11      scroll_events_90d     0.361378         0.361378
17           avg_position    -0.196103         0.196103
5              clicks_90d    -0.184882         0.184882
6           pageviews_90d     0.182019         0.182019
16                    ctr    -0.137622         0.137622


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim:
"logistic regression reaches 74% precision@50 versus 56% for the rule
baseline — an 18-point improvement over the baseline, and 20 points over
guessing (the 54% base rate). This is a real signal, not a leak."

Rewritten (safer language):
"On this grouped train/test split, logistic regression measured 74%
precision@50 versus 56% for the rule baseline — an 18-point observed gap,
and 20 points above the 54% base rate. The gap held after removing the
last_30d/prev_30d feature family, which is a directional sign against
leakage, though this is based on a single split rather than repeated
cross-validation, so the exact number should be read as decision-support,
not a guaranteed generalization to new data."

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.